# Diabetic Retinopathy Screening — Explainability Analysis

This notebook generates all global and local explanation artifacts.

**Stakeholders:**
- Global: Procurement officers at non-profit organizations
- Local: Nurses/volunteers who perform screenings with patients

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
import cv2
import os
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow version: {tf.__version__}")
print("All imports successful.")

TensorFlow version: 2.21.0
All imports successful.


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
HEIGHT, WIDTH, CHANNELS = 320, 320, 3
N_CLASSES = 5
BATCH_SIZE = 32
DATA_DIR = "data"
MODEL_PATH = "model.h5"
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

diagnosis_dict = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative DR"}

# To generate local explanations for other patients, change these IDs.
# Use id_code values from data/test.csv (without .png extension).
PATIENT_IDS = []  # Will be set after we find good examples

In [3]:
# Load CSV data
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

train_df["id_code"] = train_df["id_code"].apply(lambda x: str(x) + ".png")
test_df["id_code"] = test_df["id_code"].apply(lambda x: str(x) + ".png")
train_df["diagnosis"] = train_df["diagnosis"].astype(str)
test_df["diagnosis"] = test_df["diagnosis"].astype(str)

print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")
print(f"Columns: {list(test_df.columns)}")
print(f"\nTrain class distribution:")
print(train_df["diagnosis"].value_counts().sort_index())
print(f"\nTest head:")
print(test_df.head())

Train samples: 2929, Test samples: 733
Columns: ['id_code', 'age', 'gender', 'diagnosis']

Train class distribution:
diagnosis
0    1432
1     307
2     800
3     156
4     234
Name: count, dtype: int64

Test head:
            id_code  age gender diagnosis
0  04aef84a2cc1.png   75      F         0
1  4ad8d3ec8789.png   26      F         0
2  cb02bb47fdc5.png   71      F         0
3  c9d42d7534e0.png   70      M         2
4  d5a39339ff3d.png   19      F         2


In [4]:
# Inference-only generator — no augmentation, shuffle=False for alignment
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=os.path.join(DATA_DIR, "test_images"),
    x_col="id_code",
    y_col="diagnosis",
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    target_size=(HEIGHT, WIDTH),
    shuffle=False,
    seed=42
)

print(f"Found {test_generator.samples} valid test images")

Found 731 validated image filenames belonging to 5 classes.


Found 731 valid test images


/Users/bhuvan/Desktop/MLIP/i4-explainability-iambhuvan/venv/lib/python3.11/site-packages/keras/src/legacy/preprocessing/image.py:918: UserWarning: Found 2 invalid image filename(s) in x_col="id_code". These filename(s) will be ignored.
  warnings.warn(


In [5]:
# Load pre-trained model (compile=False to avoid optimizer version issues)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f"Model loaded. Input: {model.input_shape}, Output: {model.output_shape}")

# Find last convolutional (4D output) layer for Grad-CAM
LAST_CONV_LAYER = None
for layer in reversed(model.layers):
    try:
        if len(layer.output.shape) == 4:
            LAST_CONV_LAYER = layer.name
            break
    except:
        pass

print(f"Last conv layer for Grad-CAM: {LAST_CONV_LAYER}")

Model loaded. Input: (None, 320, 320, 3), Output: (None, 5)
Last conv layer for Grad-CAM: activation_49


---
## Global Explanations
Generate evaluation metrics and aggregate visualizations for procurement officers.

In [6]:
# Generate predictions on full test set
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

print(f"Predictions generated for {len(y_pred)} images")
print(f"True labels: {len(y_true)} images")

 1/23 ━━━━━━━━━━━━━━━━━━━━ 54s 2s/step

 2/23 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step

 3/23 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step

 4/23 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step

 5/23 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step

...

21/23 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step

22/23 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step

23/23 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step


Predictions generated for 731 images
True labels: 731 images


In [7]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_names = [diagnosis_dict[i] for i in range(N_CLASSES)]

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names,
            yticklabels=class_names, ax=axes[0])
axes[0].set_title("Confusion Matrix (Counts)", fontsize=14)
axes[0].set_ylabel("True Label")
axes[0].set_xlabel("Predicted Label")

sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names,
            yticklabels=class_names, ax=axes[1])
axes[1].set_title("Confusion Matrix (Normalized)", fontsize=14)
axes[1].set_ylabel("True Label")
axes[1].set_xlabel("Predicted Label")

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, "global_confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: artifacts/global_confusion_matrix.png")

Saved: artifacts/global_confusion_matrix.png


/var/folders/sd/09j8cjys0_775pqynqv263f40000gn/T/ipykernel_96507/3700140794.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Classification Report & Key Metrics
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
report_text = classification_report(y_true, y_pred, target_names=class_names)
kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")
accuracy = accuracy_score(y_true, y_pred)

print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(report_text)
print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"Cohen's Kappa (quadratic weighted): {kappa:.4f}")
print("=" * 60)

# Save as formatted table image
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")
report_df = pd.DataFrame(report).transpose()
report_df = report_df.round(3)
keep_rows = class_names + ["accuracy", "macro avg", "weighted avg"]
report_df = report_df.loc[[r for r in keep_rows if r in report_df.index]]

table = ax.table(cellText=report_df.values, colLabels=report_df.columns,
                 rowLabels=report_df.index, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
ax.set_title(f"Model Performance — Accuracy: {accuracy*100:.1f}% | Cohen's Kappa: {kappa:.3f}",
             fontsize=13, fontweight="bold", pad=20)
plt.savefig(os.path.join(ARTIFACTS_DIR, "global_metrics.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: artifacts/global_metrics.png")

CLASSIFICATION REPORT
                  precision    recall  f1-score   support

           No DR       1.00      0.99      0.99       372
            Mild       0.80      0.70      0.75        63
        Moderate       0.77      0.93      0.84       198
          Severe       0.74      0.46      0.57        37
Proliferative DR       0.77      0.54      0.63        61

        accuracy                           0.89       731
       macro avg       0.81      0.72      0.76       731
    weighted avg       0.89      0.89      0.88       731

Overall Accuracy: 0.8851 (88.5%)
Cohen's Kappa (quadratic weighted): 0.9130


Saved: artifacts/global_metrics.png


/var/folders/sd/09j8cjys0_775pqynqv263f40000gn/T/ipykernel_96507/1207230573.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Training Data Class Distribution
train_diag = train_df["diagnosis"].astype(int)
counts = train_diag.value_counts().sort_index()
colors = ["#4CAF50", "#FFC107", "#FF9800", "#F44336", "#9C27B0"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([diagnosis_dict[i] for i in counts.index], counts.values, color=colors)
for bar, count in zip(bars, counts.values):
    pct = count / len(train_diag) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f"{count}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=11)
ax.set_title("Training Data: Distribution of Severity Levels", fontsize=14, fontweight="bold")
ax.set_xlabel("Severity Level")
ax.set_ylabel("Number of Images")
ax.set_ylim(0, max(counts.values) * 1.2)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, "global_class_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: artifacts/global_class_distribution.png")

Saved: artifacts/global_class_distribution.png


/var/folders/sd/09j8cjys0_775pqynqv263f40000gn/T/ipykernel_96507/2173460796.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Grad-CAM Implementation
Used for both global aggregate heatmaps and individual patient explanations.

In [10]:
def compute_gradcam(model, img_array, last_conv_layer_name, pred_index=None):
    """Compute Grad-CAM heatmap for a single image."""
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    img_tensor = tf.cast(img_array, tf.float32)
    
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        # Handle both tensor and list outputs
        if isinstance(predictions, list):
            predictions = predictions[0] if len(predictions) == 1 else tf.stack(predictions)
        if isinstance(conv_outputs, list):
            conv_outputs = conv_outputs[0]
        predictions = tf.cast(predictions, tf.float32)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    grads = tape.gradient(class_channel, conv_outputs)
    if grads is None:
        print("Warning: Gradients are None, returning uniform heatmap")
        return np.ones((10, 10), dtype=np.float32) * 0.5
    
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    conv_out = conv_outputs[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    
    return heatmap.numpy()


def create_gradcam_overlay(img, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on original image."""
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = mpl_cm.jet(heatmap_resized)[:, :, :3]
    superimposed = heatmap_colored * alpha + img * (1 - alpha)
    return np.clip(superimposed, 0, 1)


print("Grad-CAM functions defined.")

Grad-CAM functions defined.


In [11]:
# Aggregate Grad-CAM per class — shows what the model generally focuses on
N_SAMPLES_PER_CLASS = 8

# Map generator class indices to our diagnosis dict
class_index_map = test_generator.class_indices  # e.g. {'0': 0, '1': 1, ...}
print(f"Generator class indices: {class_index_map}")

fig, axes = plt.subplots(2, 5, figsize=(25, 10))

for cls_idx in range(N_CLASSES):
    # The generator labels are strings, map to the generator's internal index
    gen_cls_idx = class_index_map.get(str(cls_idx), cls_idx)
    
    correct_mask = (y_true == gen_cls_idx) & (y_pred == gen_cls_idx)
    correct_indices = np.where(correct_mask)[0]
    
    if len(correct_indices) == 0:
        # Try using just predictions if no correct ones
        pred_mask = (y_pred == gen_cls_idx)
        correct_indices = np.where(pred_mask)[0]
        if len(correct_indices) == 0:
            print(f"No predictions for class {cls_idx} ({diagnosis_dict[cls_idx]}), skipping")
            axes[0, cls_idx].set_title(f"{diagnosis_dict[cls_idx]}\n(no samples)", fontsize=12)
            axes[0, cls_idx].axis("off")
            axes[1, cls_idx].axis("off")
            continue
    
    n_use = min(N_SAMPLES_PER_CLASS, len(correct_indices))
    selected = correct_indices[:n_use]
    
    heatmaps = []
    sample_img = None
    
    for idx in selected:
        fname = test_generator.filenames[idx]
        img_path = os.path.join(DATA_DIR, "test_images", os.path.basename(fname))
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (WIDTH, HEIGHT)).astype("float32") / 255.0
        
        if sample_img is None:
            sample_img = img.copy()
        
        heatmap = compute_gradcam(model, np.expand_dims(img, 0), LAST_CONV_LAYER, pred_index=gen_cls_idx)
        heatmaps.append(heatmap)
    
    if len(heatmaps) == 0 or sample_img is None:
        continue
    
    avg_heatmap = np.mean(heatmaps, axis=0)
    overlay = create_gradcam_overlay(sample_img, avg_heatmap, alpha=0.5)
    
    axes[0, cls_idx].imshow(overlay)
    axes[0, cls_idx].set_title(f"{diagnosis_dict[cls_idx]}", fontsize=14, fontweight="bold")
    axes[0, cls_idx].axis("off")
    
    axes[1, cls_idx].imshow(avg_heatmap, cmap="jet")
    axes[1, cls_idx].set_title(f"Avg attention (n={len(heatmaps)})", fontsize=11)
    axes[1, cls_idx].axis("off")
    
    fig_s, ax_s = plt.subplots(figsize=(5, 5))
    ax_s.imshow(overlay)
    ax_s.set_title(f"{diagnosis_dict[cls_idx]}", fontsize=14)
    ax_s.axis("off")
    fig_s.savefig(os.path.join(ARTIFACTS_DIR, f"global_gradcam_class_{cls_idx}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig_s)

axes[0, 0].set_ylabel("Grad-CAM Overlay", fontsize=12)
axes[1, 0].set_ylabel("Average Heatmap", fontsize=12)
fig.suptitle("What the Model Focuses On — Aggregate Grad-CAM by Severity Level", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, "global_gradcam_all_classes.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: artifacts/global_gradcam_all_classes.png and per-class images")

Generator class indices: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}


Saved: artifacts/global_gradcam_all_classes.png and per-class images


/var/folders/sd/09j8cjys0_775pqynqv263f40000gn/T/ipykernel_96507/3072242983.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Fairness Analysis — Performance by Gender and Age Group
test_df_valid = test_df.copy()
valid_filenames = [os.path.basename(f) for f in test_generator.filenames]
test_df_valid = test_df_valid[test_df_valid["id_code"].isin(valid_filenames)].reset_index(drop=True)
test_df_valid["y_true"] = y_true
test_df_valid["y_pred"] = y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By gender
gender_acc = test_df_valid.groupby("gender").apply(
    lambda g: accuracy_score(g["y_true"], g["y_pred"])
).reset_index()
gender_acc.columns = ["Gender", "Accuracy"]
bars = axes[0].bar(gender_acc["Gender"], gender_acc["Accuracy"], color=["#2196F3", "#E91E63"])
for bar, acc in zip(bars, gender_acc["Accuracy"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{acc:.3f}", ha="center", fontsize=12)
axes[0].set_title("Accuracy by Gender", fontsize=14, fontweight="bold")
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel("Accuracy")

# By age group
test_df_valid["age_group"] = pd.cut(test_df_valid["age"], bins=[0, 40, 55, 70, 100],
                                     labels=["<40", "40-55", "55-70", "70+"])
age_acc = test_df_valid.groupby("age_group", observed=True).apply(
    lambda g: accuracy_score(g["y_true"], g["y_pred"])
).reset_index()
age_acc.columns = ["Age Group", "Accuracy"]
bars2 = axes[1].bar(age_acc["Age Group"].astype(str), age_acc["Accuracy"], color="#4CAF50")
for bar, acc in zip(bars2, age_acc["Accuracy"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{acc:.3f}", ha="center", fontsize=12)
axes[1].set_title("Accuracy by Age Group", fontsize=14, fontweight="bold")
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel("Accuracy")

plt.suptitle("Fairness Analysis — Model Performance Across Demographics", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, "global_fairness.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: artifacts/global_fairness.png")

Saved: artifacts/global_fairness.png


/var/folders/sd/09j8cjys0_775pqynqv263f40000gn/T/ipykernel_96507/2557384094.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Local Explanations
Generate per-patient screening reports for nurses/volunteers.

In [13]:
def create_confidence_chart(pred_probs, save_path):
    """Create horizontal bar chart of prediction probabilities."""
    fig, ax = plt.subplots(figsize=(8, 3.5))
    classes = [diagnosis_dict[i] for i in range(N_CLASSES)]
    predicted = np.argmax(pred_probs)
    colors = ["#4CAF50" if i == predicted else "#E0E0E0" for i in range(N_CLASSES)]
    
    bars = ax.barh(classes, pred_probs, color=colors)
    for bar, prob in zip(bars, pred_probs):
        if prob > 0.02:
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{prob*100:.1f}%", va="center", fontsize=11)
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("Confidence", fontsize=12)
    ax.set_title("How Sure Is the Computer?", fontsize=13, fontweight="bold")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

print("Confidence chart function defined.")

Confidence chart function defined.


In [14]:
def generate_screening_text(prediction, confidence):
    """Generate plain-language screening explanation at 8th grade reading level."""
    severity_descriptions = {
        0: "The screening did not find signs of diabetic eye damage. This is a good result.",
        1: "The screening found mild signs of diabetic eye damage. This means there may be small changes in the blood vessels in your eye.",
        2: "The screening found moderate signs of diabetic eye damage. This means some blood vessels in your eye may be affected.",
        3: "The screening found severe signs of diabetic eye damage. This means many blood vessels in your eye may be damaged.",
        4: "The screening found advanced signs of diabetic eye damage (proliferative). This means new, fragile blood vessels may be growing in your eye."
    }
    
    next_steps_dict = {
        0: "Keep up with your regular health checkups. If you have diabetes, have your eyes checked at least once a year. Eating well and managing your blood sugar can help keep your eyes healthy.",
        1: "Please see an eye doctor for a full eye exam. Early changes were found, and a doctor can tell you more. Keep managing your blood sugar and attend regular checkups.",
        2: "Please see an eye doctor within the next few weeks. The screening found changes that need a closer look. A doctor can discuss treatment options with you.",
        3: "Please see an eye doctor as soon as possible, ideally within the next week. The screening found serious changes that may need treatment to protect your vision.",
        4: "Please see an eye doctor right away. This is urgent. The screening found advanced changes that need prompt treatment to protect your vision. Do not delay."
    }
    
    pct = confidence * 100
    if confidence > 0.8:
        conf_text = f"The computer gave this result its highest score ({pct:.0f} out of 100). This suggests the result is likely correct, but a doctor should still confirm."
    elif confidence > 0.5:
        conf_text = f"The computer gave this result a moderate score ({pct:.0f} out of 100). The computer was less certain about this result, so a doctor's confirmation is especially important."
    else:
        conf_text = f"The computer gave this result a low score ({pct:.0f} out of 100). The computer was not very certain, so please see a doctor for a full exam to confirm."
    
    return {
        "result_description": severity_descriptions[prediction],
        "confidence_text": conf_text,
        "next_steps": next_steps_dict[prediction]
    }

print("Screening text generator defined.")

Screening text generator defined.


In [15]:
def generate_patient_report(patient_id, output_dir=ARTIFACTS_DIR):
    """Generate a complete screening report for one patient."""
    os.makedirs(output_dir, exist_ok=True)
    
    patient_row = test_df[test_df["id_code"] == patient_id + ".png"]
    if len(patient_row) == 0:
        raise ValueError(f"Patient {patient_id} not found in test.csv")
    patient_row = patient_row.iloc[0]
    age = patient_row["age"]
    gender = "Female" if patient_row["gender"] == "F" else "Male"
    true_label = int(patient_row["diagnosis"])
    
    img_path = os.path.join(DATA_DIR, "test_images", patient_id + ".png")
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Image not found: {img_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (WIDTH, HEIGHT)).astype("float32") / 255.0
    
    img_batch = np.expand_dims(img_resized, 0)
    pred_probs = model.predict(img_batch, verbose=0)[0]
    prediction = int(np.argmax(pred_probs))
    confidence = float(pred_probs[prediction])
    
    heatmap = compute_gradcam(model, img_batch, LAST_CONV_LAYER, pred_index=prediction)
    overlay = create_gradcam_overlay(img_resized, heatmap, alpha=0.4)
    
    orig_path = os.path.join(output_dir, f"{patient_id}_original.png")
    gradcam_path = os.path.join(output_dir, f"{patient_id}_gradcam.png")
    confidence_path = os.path.join(output_dir, f"{patient_id}_confidence.png")
    
    plt.figure(figsize=(5, 5))
    plt.imshow(img_resized)
    plt.axis("off")
    plt.savefig(orig_path, dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close()
    
    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis("off")
    plt.savefig(gradcam_path, dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close()
    
    create_confidence_chart(pred_probs, confidence_path)
    
    texts = generate_screening_text(prediction, confidence)
    
    html = open("explanation_template.html").read()
    html = html.replace("PXX", patient_id)
    html = html.replace("AXX", str(age))
    html = html.replace("GXX", gender)
    html = html.replace("DXX", diagnosis_dict[prediction])
    html = html.replace("RESULT_DESCRIPTION", texts["result_description"])
    html = html.replace("ORIGINAL_IMG", os.path.basename(orig_path))
    html = html.replace("GRADCAM_IMG", os.path.basename(gradcam_path))
    html = html.replace("CONFIDENCE_IMG", os.path.basename(confidence_path))
    html = html.replace("CONFIDENCE_TEXT", texts["confidence_text"])
    html = html.replace("NEXT_STEPS", texts["next_steps"])
    
    html_path = os.path.join(output_dir, f"{patient_id}_report.html")
    with open(html_path, "w") as f:
        f.write(html)
    
    result = {
        "patient_id": patient_id,
        "age": age,
        "gender": gender,
        "true_label": true_label,
        "true_label_name": diagnosis_dict[true_label],
        "prediction": prediction,
        "prediction_name": diagnosis_dict[prediction],
        "confidence": confidence,
        "pred_probs": pred_probs,
        "texts": texts,
        "orig_path": orig_path,
        "gradcam_path": gradcam_path,
        "confidence_path": confidence_path,
        "html_path": html_path
    }
    
    print(f"Report generated for patient {patient_id}:")
    print(f"  True: {diagnosis_dict[true_label]}, Predicted: {diagnosis_dict[prediction]} ({confidence*100:.1f}%)")
    print(f"  Files: {html_path}")
    
    return result

print("Patient report generator defined.")

Patient report generator defined.


In [16]:
# Select two patients with different severity levels
# If PATIENT_IDS was set manually in Cell 2, skip auto-selection.
if PATIENT_IDS:
    print(f"Using manually specified patients: {PATIENT_IDS}")
else:
    valid_fnames = set(os.listdir(os.path.join(DATA_DIR, "test_images")))
    test_df_valid2 = test_df[test_df["id_code"].isin(valid_fnames)].reset_index(drop=True)

    candidates = []
    for _, row in test_df_valid2.iterrows():
        pid = row["id_code"].replace(".png", "")
        img_path = os.path.join(DATA_DIR, "test_images", row["id_code"])
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (WIDTH, HEIGHT)).astype("float32") / 255.0
        probs = model.predict(np.expand_dims(img, 0), verbose=0)[0]
        pred = int(np.argmax(probs))
        conf = float(probs[pred])
        true_label = int(row["diagnosis"])
        candidates.append({"id": pid, "true": true_label, "pred": pred, "conf": conf})
        if len(candidates) >= 100:
            break

    candidates_df = pd.DataFrame(candidates)

    # Patient A: high confidence correct prediction
    patient_a_cands = candidates_df[(candidates_df["true"] == candidates_df["pred"]) & (candidates_df["conf"] > 0.7)]
    if len(patient_a_cands[patient_a_cands["pred"] == 0]) > 0:
        patient_a = patient_a_cands[patient_a_cands["pred"] == 0].sort_values("conf", ascending=False).iloc[0]["id"]
    else:
        patient_a = patient_a_cands.sort_values("conf", ascending=False).iloc[0]["id"]

    # Patient B: moderate severity
    patient_b_cands = candidates_df[candidates_df["pred"].isin([1, 2, 3])]
    if len(patient_b_cands) > 0:
        patient_b = patient_b_cands.sort_values("conf", ascending=True).iloc[0]["id"]
    else:
        patient_b = candidates_df[candidates_df["id"] != patient_a].iloc[0]["id"]

    PATIENT_IDS = [patient_a, patient_b]
    print(f"Auto-selected patients: {PATIENT_IDS}")

Auto-selected patients: ['dd90c321d7bc', '75a4343b12f9']


In [17]:
# Generate reports for selected patients
reports = []
for pid in PATIENT_IDS:
    report = generate_patient_report(pid)
    reports.append(report)
    print()

# Copy w3.css to artifacts so HTML reports can find it
import shutil
shutil.copy("w3.css", os.path.join(ARTIFACTS_DIR, "w3.css"))
print("Copied w3.css to artifacts/")

Report generated for patient dd90c321d7bc:
  True: No DR, Predicted: No DR (100.0%)
  Files: artifacts/dd90c321d7bc_report.html



Report generated for patient 75a4343b12f9:
  True: No DR, Predicted: Mild (40.2%)
  Files: artifacts/75a4343b12f9_report.html

Copied w3.css to artifacts/


In [18]:
# Generate explanation_local.md with both patient reports
md_lines = []
md_lines.append("# Individual Screening Result Reports")
md_lines.append("")
md_lines.append("These reports are designed for **nurses and volunteers** who perform diabetic retinopathy screenings. Each report is generated after a screening and can be displayed on-screen or printed as a handout. The nurse uses this report to (1) understand the screening result, (2) explain it to the patient in plain language, and (3) decide on the appropriate referral action.")
md_lines.append("")
md_lines.append("---")
md_lines.append("")

for i, r in enumerate(reports):
    md_lines.append(f"## Patient {i+1}: {r['patient_id']}")
    md_lines.append("")
    md_lines.append(f"**Patient Information:** Age {r['age']}, {r['gender']}")
    md_lines.append("")
    md_lines.append("**Notice for the health worker:** This screening was performed by a computer program (AI), not a doctor. Please communicate this to the patient and emphasize that a doctor should confirm the result.")
    md_lines.append("")
    md_lines.append("### How This Screening Works")
    md_lines.append("*Read or paraphrase this to the patient:*")
    md_lines.append("")
    md_lines.append("A photo was taken of the back of the patient's eye using a smartphone with a special lens. A computer program analyzed the photo for signs of blood vessel damage that can be caused by diabetes.")
    md_lines.append("")
    md_lines.append("### Who Is Involved in This Decision")
    md_lines.append("")
    md_lines.append("- **You** (the health worker) took the eye photo.")
    md_lines.append("- **A computer program** analyzed it.")
    md_lines.append("- **A doctor** should confirm the result before any treatment decisions.")
    md_lines.append("")
    md_lines.append("### What Data Was Used")
    md_lines.append("")
    md_lines.append("The computer used **only the eye photo** to produce the screening result. The model does not use the patient's age or gender for its analysis. Age and gender are recorded in the patient's medical file for clinical context only.")
    md_lines.append("")
    md_lines.append("### Eye Photo and Computer Focus Area")
    md_lines.append("")
    md_lines.append(f"| Patient's eye photo | Where the computer focused |")
    md_lines.append(f"|:-:|:-:|")
    md_lines.append(f"| ![Original]({r['orig_path']}) | ![Grad-CAM]({r['gradcam_path']}) |")
    md_lines.append("")
    md_lines.append("The colored areas in the right image show where the computer looked most closely. Red and yellow indicate areas of higher attention.")
    md_lines.append("")
    md_lines.append(f"### Screening Result: {r['prediction_name']}")
    md_lines.append("")
    md_lines.append(f"{r['texts']['result_description']}")
    md_lines.append("")
    md_lines.append("### Computer Confidence")
    md_lines.append("")
    md_lines.append(f"![Confidence]({r['confidence_path']})")
    md_lines.append("")
    md_lines.append(f"{r['texts']['confidence_text']}")
    md_lines.append("")
    md_lines.append("### Recommended Action")
    md_lines.append("*Communicate the following to the patient:*")
    md_lines.append("")
    md_lines.append(f"{r['texts']['next_steps']}")
    md_lines.append("")
    md_lines.append("---")
    md_lines.append("")
    md_lines.append("**Reminder:** This is a screening tool only. It does not replace a full eye exam by a doctor. Please refer the patient to a medical professional for a complete evaluation.")
    md_lines.append("")
    md_lines.append("---")
    md_lines.append("")

with open("explanation_local.md", "w") as f:
    f.write("\n".join(md_lines))

print("Saved: explanation_local.md")

Saved: explanation_local.md


In [19]:
# Readability verification for local explanations
import textstat

patient_texts = []
for r in reports:
    patient_texts.extend([
        r["texts"]["result_description"],
        r["texts"]["confidence_text"],
        r["texts"]["next_steps"],
    ])

# Add common sections
patient_texts.extend([
    "This screening was performed by a computer program (AI), not a doctor. Please communicate this to the patient and emphasize that a doctor should confirm the result.",
    "A photo was taken of the back of the patient's eye using a smartphone with a special lens. A computer program analyzed the photo for signs of blood vessel damage that can be caused by diabetes.",
    "The computer used only the eye photo to produce the screening result. The model does not use the patient's age or gender for its analysis. Age and gender are recorded in the patient's medical file for clinical context only.",
    "This is a screening tool only. It does not replace a full eye exam by a doctor. Please refer the patient to a medical professional for a complete evaluation.",
])

combined = " ".join(patient_texts)
fk_grade = textstat.flesch_kincaid_grade(combined)
flesch_ease = textstat.flesch_reading_ease(combined)

print(f"Flesch-Kincaid Grade Level: {fk_grade:.1f}")
print(f"Flesch Reading Ease: {flesch_ease:.1f}")
print(f"Average Sentence Length: {textstat.avg_sentence_length(combined):.1f} words")
print()
if fk_grade <= 8.0:
    print(f"PASS: Grade {fk_grade:.1f} is at or below 8th grade reading level.")
else:
    print(f"WARNING: Grade {fk_grade:.1f} exceeds 8th grade target.")

Flesch-Kincaid Grade Level: 6.1
Flesch Reading Ease: 72.3
Average Sentence Length: 11.9 words

PASS: Grade 6.1 is at or below 8th grade reading level.


/var/folders/sd/09j8cjys0_775pqynqv263f40000gn/T/ipykernel_96507/297518483.py:26: DeprecationWarning: The 'avg_sentence_length' method has been deprecated due to being the same as 'words_per_sentence'. This method will be removed in thefuture.
  print(f"Average Sentence Length: {textstat.avg_sentence_length(combined):.1f} words")


---
## Summary of Generated Artifacts

All files saved to `artifacts/` directory. Markdown files at repo root.